## A2A Server with Claude - Streaming Responses Example

In [1]:
%pip install a2a-sdk==0.3.8 anthropic==0.120.2 python-dotenv uvicorn

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

print(claude_api_key)

### Setting up the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Create an Agent

In [4]:
agent = client.beta.agents.create(
    name="A2A remote agent - generic",
    model=claude_model_name,
    system="You are a helpful AI Assistant.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

Agent ID: agent_011khgHNdofA6xawbMcwYVCn, version: 1


### Create an Environment

In [5]:
environment = client.beta.environments.create(
    name="A2A remote environment - generic",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

Environment ID: env_01BP2r4DqkWxQiPz4Xv3sZZj


### Start a Session

In [6]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="A2A remote session - generic",
)

print(f"Session ID: {session.id}")

Session ID: sesn_0193GLiTZaSsNwuzdHHf2PkA


### Create the Claude Agent Class with Function Execution

In [19]:
class ClaudeRemoteAgent:
    """Helper functions for interacting with our Claude Managed Agent."""

    async def invoke_agent_stream(self, user_query: str):
        try:

            # Open the event stream
            with client.beta.sessions.events.stream(session.id) as stream:

                # Send the user's message to the agent
                client.beta.sessions.events.send(
                    session.id,
                    events=[
                        {
                            "type": "user.message",
                            "content": [
                                {
                                    "type": "text",
                                    "text": user_query,
                                },
                            ],
                        },
                    ],
                )

                # Process streaming events
                for event in stream:

                    match event.type:

                        # Agent generated a message
                        case "agent.message":
                            for block in event.content:
                                if hasattr(block, "text"):
                                    yield {
                                        "content": block.text,
                                        "done": False,
                                    }

                        # Agent is invoking a tool
                        case "agent.tool_use":
                            yield {
                                "content": f"\n[Using tool: {event.name}]\n",
                                "done": False,
                            }

                        # Agent has finished processing
                        case "session.status_idle":
                            yield {
                                "content": "",
                                "done": True,
                            }
                            break

        except Exception as e:
            print(f"Error: {e!s}")

            yield {
                "content": "Sorry, an error occurred while processing your request.",
                "done": True,
            }

### Creating the A2A Agent Executor with Streaming Responses

In [20]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_text_artifact
from a2a.types import (
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)


class ClaudeAgentExecutor(AgentExecutor):
    """Claude Agent Executor Definition."""

    def __init__(self):
        self.agent = ClaudeRemoteAgent()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()
        if not context.message:
            raise Exception('No message provided')

        # If your agent does not support streaming, just call invoke_agent
        async for event in self.agent.invoke_agent_stream(query):
            message = TaskArtifactUpdateEvent(
                context_id=context.context_id,  # type: ignore
                task_id=context.task_id,  # type: ignore
                artifact=new_text_artifact(
                    name='current_result',
                    text=event['content'],
                ),
            )
            await event_queue.enqueue_event(message)
            if event['done']:
                break

        status = TaskStatusUpdateEvent(
            context_id=context.context_id,  # type: ignore
            task_id=context.task_id,  # type: ignore
            status=TaskStatus(state=TaskState.completed),
            final=True,
        )
        await event_queue.enqueue_event(status)

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception('cancel not supported')

### Creating the Agent Skill Definition

In [21]:
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)

skill = AgentSkill(
    id = "a2a_remote_agent_skill",
    name = "Stream Responses API from a A2A remote agent",
    description = "Stream Responses API from A2A remote agent",
    tags = ["A2A remote agent"],
    examples = ["hi, how are you?", "can you tell me something about GenAI and LLMs"]
)

### Creating the Agent Card

In [22]:
public_agent_card = AgentCard(
    name = "A2A remote agent - generic",
    description = "A2A remote agent to Show A2A Usage with Claude Agent Service",
    url = "http://localhost:8080",
    version = "1.0.0",
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(streaming=True),
    skills = [skill]
)

### Creating the Request Handler

In [25]:
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

request_handler = DefaultRequestHandler(
    agent_executor = ClaudeAgentExecutor(),
    task_store = InMemoryTaskStore()
)

### Creating the A2A Server

In [28]:
from a2a.server.apps import A2AStarletteApplication

server = A2AStarletteApplication(
    agent_card = public_agent_card,
    http_handler = request_handler
)

### Starting the A2A Server

Navigate to http://localhost:8080/.well-known/agent.json to see the agent public card

In [ ]:
import asyncio
import uvicorn

config = uvicorn.Config(
    server.build(),
    host="0.0.0.0",
    port=8080,
    loop="asyncio",
)

server_instance = uvicorn.Server(config)

await server_instance.serve()

INFO:     Started server process [18511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


INFO:     127.0.0.1:51954 - "GET / HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:51954 - "GET /favicon.ico HTTP/1.1" 404 Not Found


Deprecated agent card endpoint '/.well-known/agent.json' accessed. Please use '/.well-known/agent-card.json' instead. This endpoint will be removed in a future version.


INFO:     127.0.0.1:51987 - "GET /.well-known/agent.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:51987 - "GET /favicon.ico HTTP/1.1" 404 Not Found
